Import Libraries

In [5]:
%pip install scikit-learn imbalanced-learn xgboost lightgbm joblib

Note: you may need to restart the kernel to use updated packages.


In [6]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

from sklearn.impute import SimpleImputer

from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

import joblib

Load Dataset

In [ ]:
df = pd.read_csv("final_modeling_dataset.csv")
print(df.shape)
df.head()

(32593, 49)


,code_module,code_presentation,id_student,gender,region,highest_education,imd_band,age_band,num_of_prev_attempts,studied_credits,...,quiz,repeatactivity,resource,sharedsubpage,subpage,url,course_duration,number_of_early_assessments,average_early_assessment_weight,early_assessment_workload
0,AAA,2013J,11391,M,East Anglian Region,HE Qualification,90-100%,55<=,0,240,...,0.0,0.0,9.0,0.0,23.0,1.0,268,701.0,14.878745,10430.0
1,AAA,2013J,28400,F,Scotland,HE Qualification,20-30%,35-55,0,60,...,0.0,0.0,5.0,0.0,62.0,33.0,268,701.0,14.878745,10430.0
2,AAA,2013J,30268,F,North Western Region,A Level or Equivalent,30-40%,35-55,0,60,...,0.0,0.0,4.0,0.0,22.0,4.0,268,701.0,14.878745,10430.0
3,AAA,2013J,31604,F,South East Region,A Level or Equivalent,50-60%,35-55,0,60,...,0.0,0.0,10.0,0.0,78.0,39.0,268,701.0,14.878745,10430.0
4,AAA,2013J,32885,F,West Midlands Region,Lower Than A Level,50-60%,0-35,0,60,...,0.0,0.0,7.0,0.0,29.0,6.0,268,701.0,14.878745,10430.0


Dataset Overview

In [8]:
print(df.info())
print(df.describe())
print(df.isnull().sum().sort_values(ascending=False).head(20))

<class 'pandas.DataFrame'>
RangeIndex: 32593 entries, 0 to 32592
Data columns (total 49 columns):
 #   Column                           Non-Null Count  Dtype  
---  ------                           --------------  -----  
 0   code_module                      32593 non-null  str    
 1   code_presentation                32593 non-null  str    
 2   id_student                       32593 non-null  int64  
 3   gender                           32593 non-null  str    
 4   region                           32593 non-null  str    
 5   highest_education                32593 non-null  str    
 6   imd_band                         31482 non-null  str    
 7   age_band                         32593 non-null  str    
 8   num_of_prev_attempts             32593 non-null  int64  
 9   studied_credits                  32593 non-null  int64  
 10  disability                       32593 non-null  str    
 11  dropout_target                   32593 non-null  int64  
 12  assessment_count             

Handle Missing Values

In [ ]:
#Numerical columns
numeric_cols = df.select_dtypes(include=["int64","float64"]).columns.tolist()
numeric_cols.remove("dropout_target")

In [10]:
#Categorical columns
categorical_cols = df.select_dtypes(include="object").columns.tolist()


C:\Users\jaypr\AppData\Local\Temp\ipykernel_29768\2274615285.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = df.select_dtypes(include="object").columns.tolist()


In [11]:
#Create Imputers
numeric_imputer = SimpleImputer(strategy="median")
categorical_imputer = SimpleImputer(strategy="most_frequent")

Remove Identifier Columns

In [12]:
df = df.drop(columns=["id_student"])

Separate Features and Target

In [13]:
X = df.drop(columns=["dropout_target"])
y = df["dropout_target"]
print(X.shape)
print(y.shape)

(32593, 47)
(32593,)


Identify Feature Types

In [14]:
numeric_features = X.select_dtypes(include=["int64","float64"]).columns.tolist()
categorical_features = X.select_dtypes(include="object").columns.tolist()
print(len(numeric_features))
print(len(categorical_features))

39
8


C:\Users\jaypr\AppData\Local\Temp\ipykernel_29768\1374614789.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_features = X.select_dtypes(include="object").columns.tolist()


Train-Test Split

Split before encoding and scaling to avoid data leakage.

In [28]:
# First split: Training (70%) and Temporary (30%)

X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

# Second split: Evaluation (15%) and Test (15%)

X_eval, X_test, y_eval, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp
)

print("Training Shape :", X_train.shape)
print("Evaluation Shape :", X_eval.shape)
print("Testing Shape :", X_test.shape)

print("\nTraining Distribution")
print(y_train.value_counts(normalize=True))

print("\nEvaluation Distribution")
print(y_eval.value_counts(normalize=True))

print("\nTesting Distribution")
print(y_test.value_counts(normalize=True))

Training Shape : (22815, 47)
Evaluation Shape : (4889, 47)
Testing Shape : (4889, 47)

Training Distribution
dropout_target
0    0.688407
1    0.311593
Name: proportion, dtype: float64

Evaluation Distribution
dropout_target
0    0.68828
1    0.31172
Name: proportion, dtype: float64

Testing Distribution
dropout_target
0    0.688484
1    0.311516
Name: proportion, dtype: float64


Create Preprocessing Pipeline

In [ ]:
#Numerical pipeline
numeric_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]
)

In [18]:
#Categorical pipeline
categorical_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore"))
    ]
)


In [19]:
#Combine both numerical and categorical
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_pipeline, numeric_features),
        ("cat", categorical_pipeline, categorical_features)
    ]
)

Fit on Train Only

In [29]:
# Fit preprocessing only on Training data
X_train_processed = preprocessor.fit_transform(X_train)

# Transform Evaluation and Test data
X_eval_processed = preprocessor.transform(X_eval)
X_test_processed = preprocessor.transform(X_test)
print("Training :", X_train_processed.shape)
print("Evaluation :", X_eval_processed.shape)
print("Testing :", X_test_processed.shape)

Training : (22815, 85)
Evaluation : (4889, 85)
Testing : (4889, 85)


Save Everything

In [30]:
#Save the transformed datasets
joblib.dump(preprocessor,"preprocessor.pkl")

['preprocessor.pkl']

In [31]:
# Save processed datasets

joblib.dump(X_train_processed, "X_train.pkl")
joblib.dump(X_eval_processed, "X_eval.pkl")
joblib.dump(X_test_processed, "X_test.pkl")
joblib.dump(y_train, "y_train.pkl")
joblib.dump(y_eval, "y_eval.pkl")
joblib.dump(y_test, "y_test.pkl")
joblib.dump(preprocessor, "preprocessor.pkl")


['preprocessor.pkl']

Validation

In [32]:
print("Preprocessing Completed Successfully")
print("\nTraining samples :", X_train_processed.shape)
print("Evaluation samples :", X_eval_processed.shape)
print("Testing samples :", X_test_processed.shape)
print("\nTraining Distribution")
print(y_train.value_counts())
print("\nEvaluation Distribution")
print(y_eval.value_counts())
print("\nTesting Distribution")
print(y_test.value_counts())

Preprocessing Completed Successfully

Training samples : (22815, 85)
Evaluation samples : (4889, 85)
Testing samples : (4889, 85)

Training Distribution
dropout_target
0    15706
1     7109
Name: count, dtype: int64

Evaluation Distribution
dropout_target
0    3365
1    1524
Name: count, dtype: int64

Testing Distribution
dropout_target
0    3366
1    1523
Name: count, dtype: int64
